In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
seed =123
rng = np.random.default_rng(seed)

Create GBM stock price function, since SDE for stock price under the GBM is known

In [2]:
# The Geometric Brownian motion price simulation is able to generate the price at time T directly
def gbm_price(S0, r, sigma, T, rng):
    error = rng.standard_normal()  # generate standard normal random variables
    S = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * error)
    return S

In [3]:
def black_scholes_call(S0, K, r, T, sigma):
    d1= (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2= d1 - sigma * np.sqrt(T)
    Nd1 = norm.cdf(d1)
    Nd2 = norm.cdf(d2)
    call_price = S0 * Nd1 - K * np.exp(-r * T) * Nd2
    return call_price

Define European Call payoff:

In [4]:

def compute_call_price(S_T, K, r, T):
    payoffs = np.maximum(S_T - K, 0)
    price = np.exp(-r * T) * np.mean(payoffs)
    return price


Monte Carlo GBM comparison

In [5]:
S0 = 100  # initial stock price
K = 100   # strike price
r = 0.05  # risk-free interest rate
T = 1.0   # time to maturity in years
sigma = 0.2  # volatility
n_simulations = 100000  # number of Monte Carlo simulations

mc_call_prices = np.array([])

for _ in range(n_simulations):
    # Draw innovation from standard normal distribution
    error = rng.standard_normal()  
    # Compute the drift and volatility terms
    mu = (r - 0.5 * sigma**2) * T
    vol = sigma * np.sqrt(T)
    # Simulate stock price under measure Q at time T
    S_T = S0 * np.exp(mu + vol * error)


    # Evaluate the call option payooff at maturity
    payoff = np.maximum(S_T - K, 0)
    # Discount the payoff back to present value
    call_price = np.exp(-r * T) * payoff
    mc_call_prices = np.append(mc_call_prices, call_price)

# Calculate the call prices as average discounted payoffs
mc_call_price = np.mean(mc_call_prices)
bs_call_price = black_scholes_call(S0, K, r, T, sigma)
print(f"Monte Carlo Call Price: {mc_call_price:.4f}")
print(f"Black-Scholes Call Price: {bs_call_price:.4f}")




Monte Carlo Call Price: 10.4573
Black-Scholes Call Price: 10.4506


Also, since the European options are priced based on their payoffs at time T,
one can compute the call prices in one go using single vector instead of forloop 

In [6]:
rng = np.random.default_rng(seed)
def mc_call_price(S0, K, r, T, sigma, n_simulations, rng):
    error = rng.standard_normal(n_simulations)  
    # Compute the drift and volatility terms
    mu = (r - 0.5 * sigma**2) * T
    vol = sigma * np.sqrt(T)
    # Simulate stock price under measure Q at time T
    S_T = S0 * np.exp(mu + vol * error)

    # Evaluate the call option payooff at maturity
    payoffs = np.maximum(S_T - K, 0)
    # Discount the payoff back to present value
    call_price = np.exp(-r * T) * np.mean(payoffs)
    return call_price

mc_price = mc_call_price(S0, K, r, T, sigma, n_simulations, rng)
print(f"Monte Carlo Call Price (function): {mc_price:.4f}")

Monte Carlo Call Price (function): 10.4573


The standard errors of Monte Carlo simulation is:

In [7]:
# SE of Monte Carlo simulation
SE = np.std(mc_call_prices) / np.sqrt(n_simulations)
print(f"Standard Error of Monte Carlo Call Price: {SE:.4f}")

Standard Error of Monte Carlo Call Price: 0.0466
